# 03 – Analyse de sentiment avec Spark MLlib

Ce notebook correspond à la partie **Machine Learning** du projet Mastodon.

L’objectif est d’entraîner et d’évaluer plusieurs modèles de classification de sentiment à partir du jeu de données **Sentiment140**, puis d’appliquer le meilleur modèle aux publications Mastodon collectées par le pipeline de streaming.

Le traitement suit les étapes suivantes :

**Sentiment140 → nettoyage du texte → vectorisation TF-IDF → entraînement → évaluation → prédiction des toots Mastodon → stockage dans PostgreSQL**

Deux modèles sont comparés :

- Régression logistique
- Naive Bayes multinomial

Le modèle retenu est ensuite utilisé pour prédire le sentiment des publications Mastodon en anglais.

Le modèle étant entraîné uniquement sur deux classes, **positive** et **négative**, une zone d’incertitude est également ajoutée pour les prédictions dont la probabilité est trop proche du seuil de décision.

In [1]:
import os
import sys
import time
import pyspark
import psycopg2

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    LongType,
    StringType,
    TimestampType
)

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer,
    StopWordsRemover,
    CountVectorizer,
    IDF
)
from pyspark.ml.classification import (
    LogisticRegression,
    NaiveBayes
)
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)
from pyspark.ml.functions import vector_to_array

## 1 – Environnement et sources de données

Cette première partie permet de vérifier que l’environnement d’exécution est correctement configuré avant de lancer les traitements de Machine Learning.

Le notebook s’exécute dans un conteneur Docker contenant Spark et Python.  
Les publications Mastodon sont stockées dans PostgreSQL et seront utilisées plus tard pour appliquer le modèle de sentiment entraîné.

Les vérifications suivantes permettent de confirmer :

- les versions de Python et PySpark utilisées ;
- le répertoire courant du notebook ;
- les paramètres de connexion à PostgreSQL ;
- la présence et la structure de la table `toots`.

### 1.1 – Vérification de l’environnement d’exécution

On commence par vérifier les versions de Python et de PySpark, ainsi que les variables d’environnement utilisées pour accéder à PostgreSQL.

In [2]:
print("Python :", sys.version)
print("PySpark :", pyspark.__version__)
print("Répertoire courant :", os.getcwd())

print("PGHOST :", os.getenv("PGHOST"))
print("PGPORT :", os.getenv("PGPORT"))
print("PGDATABASE :", os.getenv("PGDATABASE"))
print("PGUSER :", os.getenv("PGUSER"))

Python : 3.8.10 (default, Sep 11 2024, 16:02:53) 
[GCC 9.4.0]
PySpark : 3.5.3
Répertoire courant : /opt/spark/work-dir/notebooks
PGHOST : postgres
PGPORT : 5432
PGDATABASE : mastodon
PGUSER : spark


### 1.2 – Connexion à PostgreSQL

Les données Mastodon collectées par le pipeline de streaming sont stockées dans une base PostgreSQL.

Cette étape permet de vérifier que la connexion à la base fonctionne correctement et d’identifier les tables disponibles dans le schéma `public`.

In [3]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

tables = cursor.fetchall()

print("Connexion PostgreSQL réussie ✅")
print("\nTables disponibles :")

for table in tables:
    print("-", table[0])

cursor.close()
conn.close()

Connexion PostgreSQL réussie ✅

Tables disponibles :
- streaming_posts_per_hour
- streaming_user_stats_hourly
- toot_sentiments
- toots


### 1.3 – Structure de la table `toots`

La table `toots` constitue la source principale des données Mastodon utilisées dans ce notebook.

On vérifie ici sa structure afin d’identifier les colonnes disponibles pour l’analyse et l’application du modèle de sentiment.

In [4]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = 'toots'
    ORDER BY ordinal_position;
""")

columns = cursor.fetchall()

print("Structure de la table toots :\n")

for column_name, data_type in columns:
    print(f"{column_name:20} -> {data_type}")

cursor.close()
conn.close()

Structure de la table toots :

id                   -> bigint
toot_id              -> character varying
created_at           -> timestamp without time zone
user_id              -> character varying
username             -> character varying
content              -> text
language             -> character varying
hashtags             -> ARRAY
favourites_count     -> integer
reblogs_count        -> integer


### 1.4 – Aperçu des données disponibles

Avant d’entraîner le modèle, on vérifie le volume de données Mastodon déjà collectées ainsi que la proportion de publications en anglais.

Seules les publications en anglais seront utilisées pour l’inférence, car le jeu de données Sentiment140 utilisé pour l’entraînement contient principalement du texte en anglais.

In [5]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE language = 'en') AS english_posts,
        MIN(created_at) AS first_post,
        MAX(created_at) AS last_post
    FROM toots;
""")

result = cursor.fetchone()

print("Nombre total de toots :", result[0])
print("Toots en anglais      :", result[1])
print("Premier toot          :", result[2])
print("Dernier toot          :", result[3])

cursor.close()
conn.close()

Nombre total de toots : 95
Toots en anglais      : 81
Premier toot          : 2026-10-02 21:29:50.734192
Dernier toot          : 2026-10-08 13:45:17.489000


In [6]:
spark = (
    SparkSession.builder
    .appName("03-mastodon-sentiment")
    .getOrCreate()
)

print("Version Spark :", spark.version)
print("Application   :", spark.sparkContext.appName)
print("Spark UI      :", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 15:44:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/08 15:45:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Version Spark : 3.5.3
Application   : 03-mastodon-sentiment
Spark UI      : http://9d066d2ef4fc:4041


## 2 – Chargement du jeu de données Sentiment140

Pour entraîner un modèle supervisé de classification de sentiment, il est nécessaire de disposer de données déjà étiquetées.

Le jeu de données **Sentiment140** est utilisé comme jeu d’entraînement. Il contient 1,6 million de tweets associés à un sentiment :

- `0` : sentiment négatif ;
- `4` : sentiment positif.

Les publications Mastodon collectées dans PostgreSQL ne possèdent pas de label de sentiment connu. Elles seront donc utilisées uniquement après l’entraînement du modèle, pour réaliser les prédictions.

### 2.1 – Vérification du fichier source

Avant le chargement dans Spark, on vérifie que le fichier Sentiment140 est bien disponible dans le dossier `data`.

In [7]:
DATA_PATH = "../data/training.1600000.processed.noemoticon.csv"

print("Chemin :", os.path.abspath(DATA_PATH))
print("Fichier présent :", os.path.exists(DATA_PATH))

if os.path.exists(DATA_PATH):
    size_mb = os.path.getsize(DATA_PATH) / (1024 * 1024)
    print(f"Taille : {size_mb:.2f} Mo")

Chemin : /opt/spark/work-dir/data/training.1600000.processed.noemoticon.csv
Fichier présent : True
Taille : 227.74 Mo


### 2.2 – Définition du schéma

Le fichier CSV ne contient pas d’en-tête. Un schéma explicite est donc défini afin d’associer chaque colonne à son nom et à son type de données.

In [8]:
sentiment_schema = StructType([
    StructField("target", IntegerType(), True),
    StructField("tweet_id", LongType(), True),
    StructField("date", StringType(), True),
    StructField("flag", StringType(), True),
    StructField("user", StringType(), True),
    StructField("text", StringType(), True)
])

### 2.3 – Chargement du jeu de données avec Spark

Le fichier est ensuite chargé sous forme de DataFrame Spark en utilisant le schéma défini précédemment.

In [9]:
sentiment_raw = (
    spark.read
    .option("header", "false")
    .option("encoding", "ISO-8859-1")
    .schema(sentiment_schema)
    .csv(DATA_PATH)
)

In [10]:
sentiment_raw.printSchema()

sentiment_raw.show(5, truncate=False)

print("Nombre de lignes :", sentiment_raw.count())

root
 |-- target: integer (nullable = true)
 |-- tweet_id: long (nullable = true)
 |-- date: string (nullable = true)
 |-- flag: string (nullable = true)
 |-- user: string (nullable = true)
 |-- text: string (nullable = true)



+------+----------+----------------------------+--------+---------------+-------------------------------------------------------------------------------------------------------------------+
|target|tweet_id  |date                        |flag    |user           |text                                                                                                               |
+------+----------+----------------------------+--------+---------------+-------------------------------------------------------------------------------------------------------------------+
|0     |1467810369|Mon Apr 06 22:19:45 PDT 2009|NO_QUERY|_TheSpecialOne_|@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D|
|0     |1467810672|Mon Apr 06 22:19:49 PDT 2009|NO_QUERY|scotthamilton  |is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!    |
|0     |1467810917|Mon Apr 06 22:19:53 PDT 2009|NO

Nombre de lignes : 1600000


### 2.4 – Analyse de la variable cible

Avant de préparer les données pour le Machine Learning, on vérifie la répartition des classes présentes dans la variable `target`.

Le jeu de données Sentiment140 utilise deux valeurs :

- `0` pour un sentiment négatif ;
- `4` pour un sentiment positif.

Cette vérification permet également de contrôler si le jeu de données est équilibré entre les deux classes.

In [11]:
sentiment_raw.groupBy("target").count().orderBy("target").show()

+------+------+
|target| count|
+------+------+
|     0|800000|
|     4|800000|
+------+------+



### 2.5 – Transformation des labels

Les algorithmes de classification de Spark MLlib utilisent une colonne numérique appelée généralement `label`.

La variable `target` de Sentiment140 est donc convertie selon la règle suivante :

- `0` → `0.0` : sentiment négatif ;
- `4` → `1.0` : sentiment positif.

Seules les colonnes utiles pour l'entraînement sont conservées : le texte du tweet et son label.

In [12]:
sentiment = (
    sentiment_raw
    .select("target", "text")
    .withColumn(
        "label",
        F.when(F.col("target") == 4, F.lit(1.0))
         .otherwise(F.lit(0.0))
    )
)

In [13]:
sentiment.groupBy("label").count().orderBy("label").show()

+-----+------+
|label| count|
+-----+------+
|  0.0|800000|
|  1.0|800000|
+-----+------+



In [14]:
sentiment.show(5, truncate=False)

+------+-------------------------------------------------------------------------------------------------------------------+-----+
|target|text                                                                                                               |label|
+------+-------------------------------------------------------------------------------------------------------------------+-----+
|0     |@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D|0.0  |
|0     |is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!    |0.0  |
|0     |@Kenichan I dived many times for the ball. Managed to save 50%  The rest go out of bounds                          |0.0  |
|0     |my whole body feels itchy and like its on fire                                                                     |0.0  |
|0     |@nationwideclass no, it's not behaving at all. i'm mad. why am i here? beca

## 3 – Prétraitement du texte

Avant d'entraîner les modèles, les textes doivent être normalisés afin de réduire le bruit et de conserver les informations les plus utiles pour l'analyse de sentiment.

Le nettoyage appliqué comprend :

- le passage du texte en minuscules ;
- la conservation explicite des principales négations anglaises ;
- la suppression des URL ;
- la suppression des mentions utilisateurs ;
- la suppression de la ponctuation et des chiffres ;
- la normalisation des espaces.

Une fonction unique est utilisée afin d'appliquer exactement le même prétraitement aux données Sentiment140 et aux publications Mastodon.

In [5]:
def nettoyer_texte(df, colonne_source, colonne_sortie="clean_text"):
    """
    Nettoie une colonne textuelle avant son utilisation
    dans le pipeline de classification de sentiment.
    """

    return (
        df
        # Passage en minuscules
        .withColumn(
            colonne_sortie,
            F.lower(F.col(colonne_source))
        )

        # Conservation des principales négations anglaises
        .withColumn(
            colonne_sortie,
            F.regexp_replace(colonne_sortie, r"\bcan't\b", "can not")
        )
        .withColumn(
            colonne_sortie,
            F.regexp_replace(colonne_sortie, r"\bwon't\b", "will not")
        )
        .withColumn(
            colonne_sortie,
            F.regexp_replace(colonne_sortie, r"n't\b", " not")
        )

        # Suppression des URL
        .withColumn(
            colonne_sortie,
            F.regexp_replace(
                colonne_sortie,
                r"https?://\S+|www\.\S+",
                " "
            )
        )

        # Suppression des mentions @utilisateur
        .withColumn(
            colonne_sortie,
            F.regexp_replace(colonne_sortie, r"@\w+", " ")
        )

        # Suppression des chiffres et de la ponctuation
        .withColumn(
            colonne_sortie,
            F.regexp_replace(colonne_sortie, r"[^a-z\s]", " ")
        )

        # Normalisation des espaces
        .withColumn(
            colonne_sortie,
            F.regexp_replace(colonne_sortie, r"\s+", " ")
        )

        # Suppression des espaces en début et fin de texte
        .withColumn(
            colonne_sortie,
            F.trim(F.col(colonne_sortie))
        )
    )


sentiment_clean = nettoyer_texte(
    sentiment,
    colonne_source="text"
)

NameError: name 'sentiment' is not defined

### 3.1 – Vérification du nettoyage

Quelques exemples sont affichés afin de comparer le texte original avec sa version nettoyée.

Cette vérification permet notamment de s'assurer que les URL et mentions sont supprimées tout en conservant les négations utiles à l'analyse de sentiment.

In [16]:
sentiment_clean.select(
    "text",
    "clean_text",
    "label"
).show(10, truncate=False)

+-------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------+-----+
|text                                                                                                               |clean_text                                                                                                 |label|
+-------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------+-----+
|@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D|awww that s a bummer you shoulda got david carr of third day to do it d                                    |0.0  |
|is upset that he can't update his Facebook by texting it... and might c

### 3.2 – Tokenisation

Le texte nettoyé est découpé en mots grâce à `RegexTokenizer`.

Chaque publication est ainsi transformée en une liste de tokens qui pourra ensuite être filtrée et vectorisée.

In [17]:
tokenizer = RegexTokenizer(
    inputCol="clean_text",
    outputCol="words",
    pattern=r"\s+",
    gaps=True,
    minTokenLength=2
)

In [18]:
tokenized = tokenizer.transform(sentiment_clean)

### 3.3 – Suppression des mots vides

Les mots très fréquents qui apportent peu d'information au modèle sont supprimés à l'aide de `StopWordsRemover`.

Certaines négations comme `no`, `not` et `nor` sont volontairement conservées car elles peuvent modifier fortement le sentiment d'une phrase.

In [19]:
english_stopwords = StopWordsRemover.loadDefaultStopWords("english")

important_negations = {"no", "not", "nor"}

custom_stopwords = [
    word
    for word in english_stopwords
    if word not in important_negations
]

stopwords_remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words",
    stopWords=custom_stopwords
)

prepared = stopwords_remover.transform(tokenized)

In [20]:
prepared.select(
    "words",
    "filtered_words",
    "label"
).show(10, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------+-----+
|words                                                                                                                          |filtered_words                                                                        |label|
+-------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------+-----+
|[awww, that, bummer, you, shoulda, got, david, carr, of, third, day, to, do, it]                                               |[awww, bummer, shoulda, got, david, carr, third, day]                                 |0.0  |
|[is, upset, that, he, can, not, update, his, facebook, by, texting, it, and, might, cry, as, result, school

## 4 – Séparation des données d'entraînement et de test

Le jeu de données est séparé en deux sous-ensembles :

- **80 %** des données sont utilisées pour entraîner les modèles ;
- **20 %** des données sont conservées pour évaluer leurs performances.

L'utilisation d'une graine fixe (`seed=42`) permet de rendre la séparation reproductible.

### 4.1 – Vérification de la répartition

Après la séparation, on vérifie le nombre d'observations présentes dans chaque sous-ensemble ainsi que la distribution des classes.

Cette étape permet de s'assurer que les jeux d'entraînement et de test restent équilibrés entre les sentiments positifs et négatifs.

In [21]:
train, test = sentiment_clean.randomSplit(
    [0.8, 0.2],
    seed=42
)

train_count = train.count()
test_count = test.count()

print("Train :", train_count)
print("Test  :", test_count)
print("Total :", train_count + test_count)

Train : 1280209
Test  : 319791
Total : 1600000


In [22]:
print("TRAIN")
train.groupBy("label").count().orderBy("label").show()

print("TEST")
test.groupBy("label").count().orderBy("label").show()

TRAIN


+-----+------+
|label| count|
+-----+------+
|  0.0|639767|
|  1.0|640442|
+-----+------+

TEST


+-----+------+
|label| count|
+-----+------+
|  0.0|160233|
|  1.0|159558|
+-----+------+



## 5 – Modèle de régression logistique

Le premier modèle testé est une **régression logistique**, un algorithme de classification couramment utilisé pour les problèmes de classification binaire.

Avant l'entraînement, les textes sont transformés en variables numériques grâce à deux étapes :

- `CountVectorizer` construit une représentation basée sur la fréquence des mots ;
- `IDF` pondère ces fréquences afin de réduire l'importance des mots très fréquents et de mettre davantage en valeur les termes discriminants.

Ces étapes sont intégrées avec le prétraitement et le modèle dans un pipeline Spark MLlib.

In [23]:
count_vectorizer = CountVectorizer(
    inputCol="filtered_words",
    outputCol="tf",
    vocabSize=30000,
    minDF=5
)

idf = IDF(
    inputCol="tf",
    outputCol="features"
)

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

pipeline_lr = Pipeline(
    stages=[
        tokenizer,
        stopwords_remover,
        count_vectorizer,
        idf,
        logistic_regression
    ]
)

print("Pipeline Logistic Regression prêt ✅")

Pipeline Logistic Regression prêt ✅


### 5.1 – Entraînement du modèle

Le pipeline est entraîné sur le jeu d'entraînement.

Le temps d'exécution est également mesuré afin de pouvoir comparer non seulement les performances prédictives des modèles, mais aussi leur coût d'entraînement.

In [24]:
start = time.time()

model_lr = pipeline_lr.fit(train)

training_time_lr = time.time() - start

print(f"Temps d'entraînement : {training_time_lr:.2f} secondes")

26/10/08 15:49:30 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Temps d'entraînement : 180.13 secondes


### 5.2 – Prédictions sur le jeu de test

Une fois entraîné, le modèle est appliqué au jeu de test, qui n'a pas été utilisé pendant l'apprentissage.

Pour chaque texte, Spark fournit notamment :

- le label réel ;
- la classe prédite ;
- la probabilité associée à chaque classe.

In [25]:
predictions_lr = model_lr.transform(test)

predictions_lr.select(
    "text",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

26/10/08 15:50:20 WARN DAGScheduler: Broadcasting large task binary with size 1081.2 KiB


+----------------------------------------------------------------------------------------------------------------------------------------+-----+----------+----------------------------------------+
|text                                                                                                                                    |label|prediction|probability                             |
+----------------------------------------------------------------------------------------------------------------------------------------+-----+----------+----------------------------------------+
|        my head feels like a bowling ball                                                                                               |0.0  |1.0       |[0.40620313180291,0.59379686819709]     |
|     #canucks                                                                                                                           |0.0  |0.0       |[0.5011184610320484,0.4988815389679516] |
|     I dont li

### 5.3 – Évaluation du modèle

Les performances de la régression logistique sont évaluées sur le jeu de test à l'aide de plusieurs métriques complémentaires :

- **Accuracy** : proportion totale de prédictions correctes ;
- **F1-score** : équilibre entre précision et rappel ;
- **AUC ROC** : capacité du modèle à distinguer les classes positives et négatives.

L'utilisation de plusieurs métriques permet d'obtenir une évaluation plus complète qu'avec l'accuracy seule.

In [26]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_lr = accuracy_evaluator.evaluate(predictions_lr)
f1_lr = f1_evaluator.evaluate(predictions_lr)
auc_lr = auc_evaluator.evaluate(predictions_lr)

print(f"Accuracy : {accuracy_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC ROC  : {auc_lr:.4f}")

26/10/08 15:50:28 WARN DAGScheduler: Broadcasting large task binary with size 1089.3 KiB
26/10/08 15:50:55 WARN DAGScheduler: Broadcasting large task binary with size 1089.3 KiB
26/10/08 15:51:23 WARN DAGScheduler: Broadcasting large task binary with size 1077.3 KiB


Accuracy : 0.7890
F1-score : 0.7889
AUC ROC  : 0.8608


### 5.4 – Matrice de confusion

La matrice de confusion permet d'analyser plus précisément les erreurs du modèle en comparant les classes réelles et les classes prédites.

Elle permet notamment d'identifier :

- les prédictions négatives correctes ;
- les prédictions positives correctes ;
- les faux positifs ;
- les faux négatifs.

In [27]:
confusion_lr = (
    predictions_lr
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

confusion_lr.show()

26/10/08 15:51:57 WARN DAGScheduler: Broadcasting large task binary with size 1087.9 KiB


+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  0.0|       0.0|123414|
|  0.0|       1.0| 36819|
|  1.0|       0.0| 30652|
|  1.0|       1.0|128906|
+-----+----------+------+



26/10/08 15:52:26 WARN DAGScheduler: Broadcasting large task binary with size 1065.1 KiB


## 6 – Modèle Naive Bayes

Le second modèle testé est un **Naive Bayes multinomial**.

Cet algorithme est particulièrement adapté aux données textuelles représentées sous forme de fréquences ou de pondérations de mots.

Comme pour la régression logistique, le modèle utilise le même pipeline de prétraitement et de vectorisation afin que la comparaison entre les deux approches soit cohérente.

In [28]:
naive_bayes = NaiveBayes(
    featuresCol="features",
    labelCol="label",
    modelType="multinomial",
    smoothing=1.0
)

pipeline_nb = Pipeline(
    stages=[
        tokenizer,
        stopwords_remover,
        count_vectorizer,
        idf,
        naive_bayes
    ]
)

print("Pipeline Naive Bayes prêt ✅")

Pipeline Naive Bayes prêt ✅


### 6.1 – Entraînement du modèle

Le pipeline Naive Bayes est entraîné sur le même jeu d'entraînement que la régression logistique.

Le temps d'entraînement est mesuré afin de comparer également la rapidité des deux modèles.

In [29]:
start = time.time()

model_nb = pipeline_nb.fit(train)

training_time_nb = time.time() - start

print(f"Temps d'entraînement Naive Bayes : {training_time_nb:.2f} secondes")

Temps d'entraînement Naive Bayes : 135.28 secondes


### 6.2 – Prédictions sur le jeu de test

Le modèle entraîné est appliqué au jeu de test afin de produire les prédictions de sentiment.

Les résultats pourront ensuite être comparés à ceux obtenus avec la régression logistique.

In [30]:
predictions_nb = model_nb.transform(test)

predictions_nb.select(
    "text",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

26/10/08 15:54:43 WARN DAGScheduler: Broadcasting large task binary with size 1309.6 KiB


+----------------------------------------------------------------------------------------------------------------------------------------+-----+----------+------------------------------------------+
|text                                                                                                                                    |label|prediction|probability                               |
+----------------------------------------------------------------------------------------------------------------------------------------+-----+----------+------------------------------------------+
|        my head feels like a bowling ball                                                                                               |0.0  |0.0       |[0.9825333740595575,0.017466625940442477] |
|     #canucks                                                                                                                           |0.0  |0.0       |[0.9580854668413572,0.041914533158642735] |
|    

### 6.3 – Évaluation du modèle

Les performances de Naive Bayes sont évaluées sur le même jeu de test que la régression logistique.

On utilise les mêmes métriques principales afin de pouvoir comparer les deux modèles dans des conditions identiques :

- **Accuracy** : proportion globale de prédictions correctes ;
- **F1-score** : équilibre entre précision et rappel.

L'AUC ROC sera calculée séparément à partir des probabilités prédites afin de permettre une comparaison cohérente avec la régression logistique.

In [31]:
accuracy_nb = accuracy_evaluator.evaluate(predictions_nb)
f1_nb = f1_evaluator.evaluate(predictions_nb)

print(f"Accuracy : {accuracy_nb:.4f}")
print(f"F1-score : {f1_nb:.4f}")

26/10/08 15:54:49 WARN DAGScheduler: Broadcasting large task binary with size 1317.7 KiB
26/10/08 15:55:16 WARN DAGScheduler: Broadcasting large task binary with size 1317.7 KiB


Accuracy : 0.7632
F1-score : 0.7632


### 6.4 – Matrice de confusion

La matrice de confusion permet d'observer la répartition des prédictions correctes et incorrectes de Naive Bayes.

Elle sera utilisée avec les autres métriques pour comparer ce modèle à la régression logistique.

In [32]:
confusion_nb = (
    predictions_nb
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

confusion_nb.show()

26/10/08 15:55:50 WARN DAGScheduler: Broadcasting large task binary with size 1316.4 KiB


+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  0.0|       0.0|122483|
|  0.0|       1.0| 37750|
|  1.0|       0.0| 37968|
|  1.0|       1.0|121590|
+-----+----------+------+



26/10/08 15:56:23 WARN DAGScheduler: Broadcasting large task binary with size 1293.5 KiB


### 6.5 – Comparaison des AUC ROC

L'AUC ROC est calculée à partir des probabilités produites par chaque modèle.

Cette métrique permet d'évaluer leur capacité à séparer les sentiments positifs et négatifs indépendamment du seuil de classification utilisé.

In [33]:
auc_probability_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="probability",
    metricName="areaUnderROC"
)

auc_lr_probability = auc_probability_evaluator.evaluate(predictions_lr)
auc_nb_probability = auc_probability_evaluator.evaluate(predictions_nb)

print(f"AUC Logistic Regression : {auc_lr_probability:.4f}")
print(f"AUC Naive Bayes          : {auc_nb_probability:.4f}")

26/10/08 15:56:23 WARN DAGScheduler: Broadcasting large task binary with size 1081.6 KiB
26/10/08 15:56:59 WARN DAGScheduler: Broadcasting large task binary with size 1309.9 KiB


AUC Logistic Regression : 0.8608
AUC Naive Bayes          : 0.8314


## 7 – Comparaison et choix du modèle final

Les deux modèles entraînés sont maintenant comparés à partir de leurs performances obtenues sur le même jeu de test.

La régression logistique obtient de meilleurs résultats que Naive Bayes sur les principales métriques d'évaluation :

- une meilleure accuracy ;
- un meilleur F1-score ;
- une meilleure AUC ROC.

Naive Bayes présente cependant un temps d'entraînement plus faible.

Dans ce projet, la qualité des prédictions est prioritaire par rapport au gain de temps d'entraînement. La **régression logistique** est donc retenue comme modèle final pour l'analyse des publications Mastodon.

### Résultats de la comparaison

| Modèle | Accuracy | F1-score | AUC ROC |
|---|---:|---:|---:|
| Régression logistique | 0,7887 | 0,7886 | 0,8606 |
| Naive Bayes | 0,7630 | 0,7630 | 0,8313 |

La régression logistique offre les meilleures performances globales et sera donc utilisée pour l'inférence sur les données Mastodon.

## 8 – Sauvegarde du modèle final

La régression logistique ayant été retenue comme meilleur modèle, le pipeline entraîné est sauvegardé afin de pouvoir être réutilisé sans avoir à relancer l'entraînement.

Le modèle sauvegardé contient l'ensemble du pipeline :

- tokenisation ;
- suppression des mots vides ;
- vectorisation du texte ;
- pondération TF-IDF ;
- modèle de régression logistique.

Cette sauvegarde permet de conserver un modèle prêt à être appliqué sur les nouvelles publications Mastodon.

In [34]:
MODEL_PATH = "../models/sentiment_lr"

os.makedirs("../models", exist_ok=True)

model_lr.write().overwrite().save(MODEL_PATH)

print("Modèle sauvegardé ✅")
print("Chemin :", os.path.abspath(MODEL_PATH))

Modèle sauvegardé ✅
Chemin : /opt/spark/work-dir/models/sentiment_lr


## 9 – Application du modèle aux publications Mastodon

Le modèle retenu est maintenant appliqué aux publications Mastodon collectées par le pipeline de streaming.

Seules les publications en anglais sont sélectionnées, car le modèle a été entraîné à partir du jeu de données Sentiment140, lui-même principalement composé de textes en anglais.

Les publications sont récupérées depuis la table PostgreSQL `toots`, puis transformées en DataFrame Spark afin de pouvoir utiliser directement le pipeline MLlib entraîné.

In [35]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        toot_id,
        created_at,
        username,
        language,
        content
    FROM toots
    WHERE language = 'en'
      AND content IS NOT NULL
    ORDER BY created_at;
""")

rows = cursor.fetchall()

cursor.close()
conn.close()

print("Toots anglais récupérés :", len(rows))

Toots anglais récupérés : 81


### 9.1 – Conversion des données PostgreSQL en DataFrame Spark

Les lignes récupérées depuis PostgreSQL sont converties en DataFrame Spark avec un schéma explicite.

Cette conversion permet d'appliquer ensuite exactement le même prétraitement et le même modèle que ceux utilisés pendant la phase d'entraînement.

In [36]:
toots_schema = StructType([
    StructField("toot_id", StringType(), False),
    StructField("created_at", TimestampType(), True),
    StructField("username", StringType(), True),
    StructField("language", StringType(), True),
    StructField("content", StringType(), True)
])

toots_df = spark.createDataFrame(rows, schema=toots_schema)

toots_df.show(5, truncate=False)

+-------+--------------------------+--------+--------+---------------------------------------+
|toot_id|created_at                |username|language|content                                |
+-------+--------------------------+--------+--------+---------------------------------------+
|1005   |2026-10-02 21:29:50.734192|bob     |en      |Data engineering project with Mastodon |
|1003   |2026-10-03 21:29:50.734192|alice   |en      |AI and machine learning are interesting|
|1004   |2026-10-03 21:29:50.734192|charlie |en      |PostgreSQL works great with Spark      |
|1002   |2026-10-04 20:29:50.734192|bob     |en      |Learning Kafka with Spark streaming    |
|1001   |2026-10-04 21:29:50.734192|alice   |en      |I love Apache Spark and Data Science   |
+-------+--------------------------+--------+--------+---------------------------------------+
only showing top 5 rows



### 9.2 – Prétraitement des publications Mastodon

Les publications Mastodon sont nettoyées avec la même fonction `nettoyer_texte()` que celle utilisée pour Sentiment140.

Cela garantit que les données utilisées pour l'inférence suivent exactement les mêmes règles de préparation que les données ayant servi à entraîner le modèle.

In [37]:
toots_clean = nettoyer_texte(
    toots_df,
    colonne_source="content"
)

### 9.3 – Prédiction des sentiments

Le pipeline de régression logistique entraîné précédemment est appliqué aux publications Mastodon prétraitées.

Pour chaque publication, le modèle fournit notamment :

- une classe prédite (`prediction`) ;
- un vecteur de probabilités (`probability`) indiquant la probabilité associée à chaque classe.

La classe `0` correspond à un sentiment négatif et la classe `1` à un sentiment positif.

In [38]:
mastodon_predictions = model_lr.transform(toots_clean)

In [39]:
mastodon_predictions.select(
    "toot_id",
    "username",
    "content",
    "prediction",
    "probability"
).show(20, truncate=False)

26/10/08 15:57:54 WARN DAGScheduler: Broadcasting large task binary with size 1066.7 KiB
26/10/08 15:57:54 WARN DAGScheduler: Broadcasting large task binary with size 1066.7 KiB


+------------------+------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+------------------------------------------+
|toot_id           |username                            |content                                                                                                                                                                                        

### 9.4 – Mise en forme des prédictions

Afin de faciliter l'interprétation des résultats, la probabilité associée à la classe positive est extraite dans une colonne `p_positive`.

Une colonne textuelle `predicted_sentiment` est également créée pour remplacer les valeurs numériques du modèle par des libellés plus lisibles :

- `0` → `negative`
- `1` → `positive`

In [40]:
from pyspark.ml.functions import vector_to_array

mastodon_results = (
    mastodon_predictions
    .withColumn(
        "p_positive",
        vector_to_array("probability")[1]
    )
    .withColumn(
        "predicted_sentiment",
        F.when(F.col("prediction") == 1.0, "positive")
         .otherwise("negative")
    )
)

In [41]:
mastodon_results.select(
    "toot_id",
    "username",
    "content",
    "predicted_sentiment",
    F.round("p_positive", 3).alias("p_positive")
).show(20, truncate=False)

26/10/08 15:57:55 WARN DAGScheduler: Broadcasting large task binary with size 1070.3 KiB
26/10/08 15:57:55 WARN DAGScheduler: Broadcasting large task binary with size 1070.3 KiB


+------------------+------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+----------+
|toot_id           |username                            |content                                                                                                                                                                                                               

### 9.5 – Gestion des prédictions incertaines

Le modèle a été entraîné uniquement sur deux classes : positive et négative.

Cependant, certaines prédictions ont une probabilité proche de `0,5`, ce qui signifie que le modèle est peu sûr de sa décision.

Pour éviter de présenter ces résultats comme certains, une zone d'incertitude est ajoutée :

- `p_positive ≤ 0,35` → négatif ;
- `0,35 < p_positive < 0,65` → incertain ;
- `p_positive ≥ 0,65` → positif.

La valeur `uncertain` ne correspond donc pas à une troisième classe apprise par le modèle, mais à une règle d'affichage basée sur la confiance de la prédiction.

In [42]:
mastodon_results = (
    mastodon_results
    .withColumn(
        "sentiment_display",
        F.when(F.col("p_positive") >= 0.65, "positive")
         .when(F.col("p_positive") <= 0.35, "negative")
         .otherwise("uncertain")
    )
)

In [43]:
mastodon_results.groupBy(
    "predicted_sentiment"
).count().orderBy("predicted_sentiment").show()

26/10/08 15:57:55 WARN DAGScheduler: Broadcasting large task binary with size 1078.9 KiB


+-------------------+-----+
|predicted_sentiment|count|
+-------------------+-----+
|           negative|   14|
|           positive|   67|
+-------------------+-----+



26/10/08 15:57:56 WARN DAGScheduler: Broadcasting large task binary with size 1060.2 KiB


In [44]:
mastodon_results.groupBy(
    "sentiment_display"
).count().orderBy("sentiment_display").show()

26/10/08 15:57:57 WARN DAGScheduler: Broadcasting large task binary with size 1084.2 KiB


+-----------------+-----+
|sentiment_display|count|
+-----------------+-----+
|         negative|    8|
|         positive|   63|
|        uncertain|   10|
+-----------------+-----+



26/10/08 15:57:57 WARN DAGScheduler: Broadcasting large task binary with size 1062.7 KiB


In [45]:
mastodon_results.select(
    "content",
    "predicted_sentiment",
    F.round("p_positive", 3).alias("p_positive"),
    "sentiment_display"
).filter(
    F.col("sentiment_display") == "uncertain"
).show(10, truncate=False)

26/10/08 15:57:58 WARN DAGScheduler: Broadcasting large task binary with size 1116.0 KiB
26/10/08 15:57:58 WARN DAGScheduler: Broadcasting large task binary with size 1116.0 KiB
26/10/08 15:57:59 WARN DAGScheduler: Broadcasting large task binary with size 1116.0 KiB


+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+----------+-----------------+
|content                                                                                                                                                                                                                                                                                                                                                                                                                             

### 9.6 – Exclusion des données de démonstration

La base PostgreSQL contient quelques publications ajoutées manuellement lors de l'initialisation du projet afin de tester le pipeline.

Ces enregistrements ne correspondent pas à de véritables publications collectées depuis Mastodon.

Afin de ne pas fausser l'analyse finale, les identifiants de démonstration `1001` à `1005` sont exclus des résultats utilisés pour l'analyse de sentiment.

In [46]:
demo_toot_ids = ["1001", "1002", "1003", "1004", "1005"]

mastodon_real_results = (
    mastodon_results
    .filter(~F.col("toot_id").isin(demo_toot_ids))
)

### 9.7 – Vérification du jeu de données final

On vérifie le nombre total de prédictions ainsi que le nombre de publications réellement issues de Mastodon après exclusion des données de démonstration.

In [47]:
print("Résultats totaux :", mastodon_results.count())
print("Vrais toots Mastodon :", mastodon_real_results.count())

Résultats totaux : 81
Vrais toots Mastodon : 76


### 9.8 – Répartition finale des sentiments

La distribution finale des sentiments est calculée uniquement sur les publications réellement collectées depuis Mastodon.

Cette répartition permet d'obtenir une première vue globale du sentiment détecté dans les publications analysées.

In [48]:
mastodon_real_results.groupBy(
    "sentiment_display"
).count().orderBy("sentiment_display").show()

26/10/08 15:58:01 WARN DAGScheduler: Broadcasting large task binary with size 1087.1 KiB


+-----------------+-----+
|sentiment_display|count|
+-----------------+-----+
|         negative|    8|
|         positive|   60|
|        uncertain|    8|
+-----------------+-----+



26/10/08 15:58:02 WARN DAGScheduler: Broadcasting large task binary with size 1064.0 KiB


## 10 – Stockage des résultats dans PostgreSQL

Les prédictions de sentiment sont enregistrées dans une table dédiée afin de séparer les données brutes Mastodon des résultats produits par le modèle de Machine Learning.

La table `toot_sentiments` contient notamment :

- l'identifiant du toot ;
- la classe prédite ;
- le sentiment affiché ;
- la probabilité d'appartenance à la classe positive ;
- le nom du modèle utilisé ;
- la date de génération de la prédiction.

Une clé étrangère relie chaque prédiction à la table `toots`.

In [49]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS toot_sentiments (
        toot_id VARCHAR(100) PRIMARY KEY,
        predicted_label SMALLINT NOT NULL,
        predicted_sentiment VARCHAR(20) NOT NULL,
        sentiment_display VARCHAR(20) NOT NULL,
        p_positive DOUBLE PRECISION NOT NULL,
        model_name VARCHAR(100) NOT NULL,
        predicted_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

        CONSTRAINT fk_toot_sentiment
            FOREIGN KEY (toot_id)
            REFERENCES toots(toot_id)
            ON DELETE CASCADE
    );
""")

conn.commit()

cursor.close()
conn.close()

print("Table toot_sentiments créée ✅")

Table toot_sentiments créée ✅


In [50]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

for table in cursor.fetchall():
    print(table[0])

cursor.close()
conn.close()

streaming_posts_per_hour
streaming_user_stats_hourly
toot_sentiments
toots


### 10.1 – Préparation des résultats à enregistrer

Les résultats Spark sont convertis dans un format compatible avec PostgreSQL.

Chaque ligne contient les informations nécessaires pour enregistrer la prédiction produite par le modèle final.

In [51]:
results_to_insert = (
    mastodon_real_results
    .select(
        "toot_id",
        F.col("prediction").cast("int").alias("predicted_label"),
        "predicted_sentiment",
        "sentiment_display",
        "p_positive"
    )
    .collect()
)

rows_to_insert = [
    (
        row["toot_id"],
        row["predicted_label"],
        row["predicted_sentiment"],
        row["sentiment_display"],
        float(row["p_positive"]),
        "logistic_regression_tfidf_sentiment140_v1"
    )
    for row in results_to_insert
]

print("Résultats prêts à être enregistrés :", len(rows_to_insert))

26/10/08 15:58:02 WARN DAGScheduler: Broadcasting large task binary with size 1072.8 KiB


Résultats prêts à être enregistrés : 76


### 10.2 – Insertion et mise à jour des prédictions

Les prédictions sont enregistrées dans PostgreSQL à l'aide d'un mécanisme d'**UPSERT**.

Grâce à la clause `ON CONFLICT`, si une publication possède déjà une prédiction, celle-ci est mise à jour au lieu de créer un doublon.

Cette approche permet de relancer le notebook plusieurs fois de manière sûre.

In [52]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.executemany("""
    INSERT INTO toot_sentiments (
        toot_id,
        predicted_label,
        predicted_sentiment,
        sentiment_display,
        p_positive,
        model_name
    )
    VALUES (%s, %s, %s, %s, %s, %s)

    ON CONFLICT (toot_id)
    DO UPDATE SET
        predicted_label = EXCLUDED.predicted_label,
        predicted_sentiment = EXCLUDED.predicted_sentiment,
        sentiment_display = EXCLUDED.sentiment_display,
        p_positive = EXCLUDED.p_positive,
        model_name = EXCLUDED.model_name,
        predicted_at = CURRENT_TIMESTAMP;
""", rows_to_insert)

conn.commit()

cursor.close()
conn.close()

print("Prédictions enregistrées dans PostgreSQL ✅")

Prédictions enregistrées dans PostgreSQL ✅


### 10.3 – Vérification du stockage

Une dernière requête permet de vérifier que les prédictions ont bien été enregistrées dans PostgreSQL et que leur distribution correspond aux résultats calculés précédemment.

In [53]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        sentiment_display,
        COUNT(*)
    FROM toot_sentiments
    GROUP BY sentiment_display
    ORDER BY sentiment_display;
""")

print("Résultats enregistrés :")

for sentiment_name, count in cursor.fetchall():
    print(f"{sentiment_name:10} : {count}")

cursor.close()
conn.close()

Résultats enregistrés :
negative   : 8
positive   : 60
uncertain  : 8


## Conclusion

Ce notebook a permis de construire une chaîne complète d'analyse de sentiment avec Spark MLlib.

Le traitement réalisé comprend :

- le chargement et la préparation du jeu de données Sentiment140 ;
- le nettoyage et la tokenisation des textes ;
- la vectorisation avec TF-IDF ;
- l'entraînement de plusieurs modèles de classification ;
- la comparaison entre la régression logistique et Naive Bayes ;
- la sélection de la régression logistique comme modèle final ;
- l'application du modèle aux publications Mastodon en anglais ;
- l'ajout d'une zone d'incertitude pour les prédictions peu confiantes ;
- le stockage des résultats dans PostgreSQL.

La régression logistique a été retenue car elle obtient les meilleures performances globales parmi les modèles testés.

### Limites

Plusieurs limites doivent cependant être prises en compte.

Le modèle est entraîné à partir de Sentiment140, un jeu de données composé de tweets. Les publications Mastodon analysées dans ce projet peuvent avoir un style différent, notamment lorsqu'elles proviennent de flux d'actualité, de comptes automatisés ou de contenus techniques.

Le modèle ne connaît également que deux classes pendant l'entraînement : positive et négative. La catégorie `uncertain` ajoutée dans ce notebook ne correspond donc pas à une troisième classe apprise, mais à une zone de confiance définie autour du seuil de décision.

Enfin, le modèle est appliqué uniquement aux publications en anglais, afin de rester cohérent avec les données utilisées pour son entraînement.

Une amélioration future pourrait consister à constituer un jeu de publications Mastodon annotées manuellement afin d'évaluer plus précisément les performances du modèle sur les données réellement utilisées dans le projet.

## Bonus – Mise à jour des prédictions sur de nouvelles publications

Après l'entraînement et l'évaluation du modèle de sentiment, le pipeline de streaming a continué à collecter de nouvelles publications Mastodon.

Ces nouvelles données n'étaient donc pas présentes lors de la première application du modèle et ne possédaient pas encore de prédiction dans la table `toot_sentiments`.

Afin d'éviter de réentraîner inutilement le modèle sur Sentiment140, le modèle Logistic Regression sauvegardé précédemment est rechargé puis appliqué uniquement aux nouvelles publications anglaises qui ne possèdent pas encore de prédiction.

In [1]:
import os

MODEL_PATH = "../models/sentiment_lr"

print("Modèle présent :", os.path.exists(MODEL_PATH))
print("Chemin :", os.path.abspath(MODEL_PATH))

Modèle présent : True
Chemin : /opt/spark/work-dir/models/sentiment_lr


In [2]:
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel

# Récupération ou création de la session Spark
spark = SparkSession.builder.getOrCreate()

# Chargement du modèle déjà entraîné
MODEL_PATH = "../models/sentiment_lr"

modele_sentiment = PipelineModel.load(MODEL_PATH)

print("Modèle chargé avec succès.")
print("Nombre d'étapes dans le pipeline :", len(modele_sentiment.stages))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 10:14:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/09 10:14:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Modèle chargé avec succès.
Nombre d'étapes dans le pipeline : 5


### Sélection des publications non encore analysées

Une requête PostgreSQL permet d'identifier uniquement les publications :

- écrites en anglais ;
- provenant réellement de Mastodon, en excluant les données de démonstration ;
- possédant un contenu textuel ;
- absentes de la table `toot_sentiments`.

Cette approche permet de réaliser une **prédiction incrémentale** : seules les nouvelles données sont traitées, tandis que les prédictions déjà enregistrées sont conservées.

In [3]:
import os
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

requete = """
    SELECT
        t.toot_id,
        t.created_at,
        t.username,
        t.language,
        t.content
    FROM toots t
    LEFT JOIN toot_sentiments s
        ON t.toot_id = s.toot_id
    WHERE t.language = 'en'
      AND t.content IS NOT NULL
      AND t.toot_id NOT IN ('1001', '1002', '1003', '1004', '1005')
      AND s.toot_id IS NULL
    ORDER BY t.created_at;
"""

cursor = conn.cursor()
cursor.execute(requete)

colonnes = [
    description[0]
    for description in cursor.description
]

donnees = cursor.fetchall()

nouveaux_toots = pd.DataFrame(
    donnees,
    columns=colonnes
)

cursor.close()
conn.close()

print("Nombre de toots à prédire :", len(nouveaux_toots))

display(nouveaux_toots.head())

Nombre de toots à prédire : 130


,toot_id,created_at,username,language,content
0,117410303720656367,2026-10-08 16:54:21.000,aftermath.site@web.brid.gy,en,Fortnite Will Give Kids What They've Always Wa...
1,117410310929277644,2026-10-09 06:57:33.000,cornucopia.se@web.brid.gy,en,"Blandad veckoavslutning i Asien, starkare kron..."
2,117409780091961911,2026-10-09 07:09:46.000,hackaday@www.urbanmind.net,en,Anatomy of an autonomy bug: when two valid dec...
3,117409813684520927,2026-10-09 07:15:02.000,hackaday@www.urbanmind.net,en,Autonomous agent payments went from demo to pr...
4,117409777674864550,2026-10-09 07:30:31.446,top_news,en,Donald Trump Has a New Name for AI & a Warning...


In [6]:
print(
    "Fonction nettoyer_texte disponible :",
    "nettoyer_texte" in globals()
)

Fonction nettoyer_texte disponible : True


### Réutilisation du pipeline de Machine Learning

Le modèle sauvegardé est chargé avec `PipelineModel.load()`.

Les nouvelles publications sont préparées avec la même fonction de nettoyage que celle utilisée lors de l'entraînement afin de conserver une cohérence entre les données d'apprentissage et les données de production.

Le pipeline applique ensuite successivement les étapes de prétraitement et le modèle de régression logistique pour produire :

- la classe prédite : positive ou négative ;
- la probabilité associée au sentiment positif ;
- une catégorie d'affichage `positive`, `negative` ou `uncertain`.

La catégorie `uncertain` ne constitue pas une troisième classe apprise par le modèle. Elle correspond à une zone de confiance intermédiaire définie à partir de la probabilité prédite.

In [8]:
from pyspark.sql import functions as F

# Conversion du DataFrame pandas en DataFrame Spark
nouveaux_toots_spark = spark.createDataFrame(nouveaux_toots)

# Application du même nettoyage que pour les données d'entraînement
nouveaux_toots_spark = nettoyer_texte(
    nouveaux_toots_spark,
    colonne_source="content",
    colonne_sortie="clean_text"
)

print(
    "Nombre de lignes préparées :",
    nouveaux_toots_spark.count()
)

nouveaux_toots_spark.select(
    "toot_id",
    "content",
    "clean_text"
).show(
    5,
    truncate=80
)

Nombre de lignes préparées : 130
+------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|           toot_id|                                                                         content|                                                                      clean_text|
+------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|117410303720656367|Fortnite Will Give Kids What They've Always Wanted: AI Pride And Prejudice Fa...|fortnite will give kids what they ve always wanted ai pride and prejudice fan...|
|117410310929277644|Blandad veckoavslutning i Asien, starkare krona – mycket hög svensk ekonomisk...|blandad veckoavslutning i asien starkare krona mycket h g svensk ekonomisk ti...|
|117409780091961911|Anatomy of an autonomy bug: when

In [9]:
from pyspark.ml.functions import vector_to_array

# Application du modèle aux nouveaux toots
predictions_nouveaux = modele_sentiment.transform(
    nouveaux_toots_spark
)

# Récupération de la probabilité de sentiment positif
predictions_nouveaux = predictions_nouveaux.withColumn(
    "p_positive",
    vector_to_array(F.col("probability"))[1]
)

# Classe prédite par le modèle
predictions_nouveaux = predictions_nouveaux.withColumn(
    "predicted_sentiment",
    F.when(
        F.col("prediction") == 1.0,
        "positive"
    ).otherwise("negative")
)

# Application de notre seuil de confiance
predictions_nouveaux = predictions_nouveaux.withColumn(
    "sentiment_display",
    F.when(
        F.col("p_positive") >= 0.65,
        "positive"
    )
    .when(
        F.col("p_positive") <= 0.35,
        "negative"
    )
    .otherwise("uncertain")
)

print(
    "Nombre de prédictions :",
    predictions_nouveaux.count()
)

predictions_nouveaux.select(
    "toot_id",
    "username",
    "p_positive",
    "predicted_sentiment",
    "sentiment_display"
).show(
    10,
    truncate=False
)

Nombre de prédictions : 130


26/10/09 10:22:24 WARN DAGScheduler: Broadcasting large task binary with size 1069.6 KiB


+------------------+------------------------------+------------------+-------------------+-----------------+
|toot_id           |username                      |p_positive        |predicted_sentiment|sentiment_display|
+------------------+------------------------------+------------------+-------------------+-----------------+
|117410303720656367|aftermath.site@web.brid.gy    |0.7996436676320974|positive           |positive         |
|117410310929277644|cornucopia.se@web.brid.gy     |0.6221231870098514|positive           |uncertain        |
|117409780091961911|hackaday@www.urbanmind.net    |0.8004349795267949|positive           |positive         |
|117409813684520927|hackaday@www.urbanmind.net    |0.7379593300853505|positive           |positive         |
|117409777674864550|top_news                      |0.9355577564286726|positive           |positive         |
|117409778394130804|RunRichRun                    |0.932889444767961 |positive           |positive         |
|117409785011037420

In [10]:
predictions_nouveaux.groupBy(
    "sentiment_display"
).count().orderBy(
    F.desc("count")
).show()

26/10/09 10:22:41 WARN DAGScheduler: Broadcasting large task binary with size 1082.4 KiB
26/10/09 10:22:42 WARN DAGScheduler: Broadcasting large task binary with size 1061.0 KiB


+-----------------+-----+
|sentiment_display|count|
+-----------------+-----+
|         positive|   96|
|         negative|   22|
|        uncertain|   12|
+-----------------+-----+



In [11]:
# Préparation des résultats à enregistrer dans PostgreSQL
resultats_nouveaux = predictions_nouveaux.select(
    "toot_id",
    "prediction",
    "predicted_sentiment",
    "sentiment_display",
    "p_positive"
).toPandas()

print("Nombre de résultats à enregistrer :", len(resultats_nouveaux))

26/10/09 10:23:22 WARN DAGScheduler: Broadcasting large task binary with size 1069.0 KiB


Nombre de résultats à enregistrer : 130


In [12]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

requete_insert = """
    INSERT INTO toot_sentiments (
        toot_id,
        predicted_label,
        predicted_sentiment,
        sentiment_display,
        p_positive,
        model_name
    )
    VALUES (%s, %s, %s, %s, %s, %s)
    ON CONFLICT (toot_id)
    DO UPDATE SET
        predicted_label = EXCLUDED.predicted_label,
        predicted_sentiment = EXCLUDED.predicted_sentiment,
        sentiment_display = EXCLUDED.sentiment_display,
        p_positive = EXCLUDED.p_positive,
        model_name = EXCLUDED.model_name,
        predicted_at = CURRENT_TIMESTAMP;
"""

donnees_insert = []

for _, row in resultats_nouveaux.iterrows():
    donnees_insert.append(
        (
            str(row["toot_id"]),
            int(row["prediction"]),
            str(row["predicted_sentiment"]),
            str(row["sentiment_display"]),
            float(row["p_positive"]),
            "logistic_regression_tfidf_sentiment140_v1"
        )
    )

cursor.executemany(
    requete_insert,
    donnees_insert
)

conn.commit()

print(
    f"{cursor.rowcount} prédictions enregistrées dans PostgreSQL."
)

cursor.close()
conn.close()

130 prédictions enregistrées dans PostgreSQL.


### Résultat de la mise à jour

Au total, **130 nouvelles publications** ont été analysées sans réentraîner le modèle.

La répartition obtenue est :

- **96 publications positives** ;
- **22 publications négatives** ;
- **12 publications incertaines**.

Les résultats sont ensuite enregistrés dans la table `toot_sentiments` avec une opération `UPSERT`, ce qui permet d'ajouter les nouvelles prédictions tout en évitant les doublons.

Cette étape illustre un fonctionnement plus proche d'un environnement réel : le modèle est entraîné une première fois, sauvegardé, puis réutilisé pour analyser progressivement de nouvelles données collectées par le pipeline.